In [ ]:
# Load merged data from Drive
import pandas as pd
import numpy as np
from google.colab import drive

drive.mount('/content/drive')

# Update path to your merged file location
merged_path = '/content/drive/MyDrive/Spring 2026/Machine Learning For Cities/MLC Final Project/data/merged_data.csv'
df = pd.read_csv(merged_path)
print(f'Loaded: {df.shape[0]:,} rows x {df.shape[1]} columns')

Mounted at /content/drive
Loaded: 2,324 rows x 53 columns


In [ ]:
# Normalize PLUTO total columns into tract-level rates

# Average building area per residential unit
df['pluto_bldgarea_per_unit'] = (
    df['pluto_total_bldgarea'] / df['pluto_total_unitsres']
)

# Average residential floor area per unit
df['pluto_resarea_per_unit'] = (
    df['pluto_total_resarea'] / df['pluto_total_unitsres']
)

# Average lot area per parcel
df['pluto_avg_lotarea'] = (
    df['pluto_total_lotarea'] / df['pluto_parcel_count']
)

# Commercial share of total floor area (mixed-use indicator)
df['pluto_commercial_share'] = (
    df['pluto_total_comarea'] / df['pluto_total_bldgarea']
)

# Replace any infinities from division by zero with NaN
for col in ['pluto_bldgarea_per_unit', 'pluto_resarea_per_unit',
            'pluto_avg_lotarea', 'pluto_commercial_share']:
    df[col] = df[col].replace([np.inf, -np.inf], np.nan)

print('New normalized PLUTO features:')
print(df[['pluto_bldgarea_per_unit', 'pluto_resarea_per_unit',
          'pluto_avg_lotarea', 'pluto_commercial_share']].describe())

New normalized PLUTO features:
       pluto_bldgarea_per_unit  pluto_resarea_per_unit  pluto_avg_lotarea  \
count              2205.000000             2205.000000        2228.000000   
mean                 10.132824               15.830461           6.494610   
std                  26.301779               33.127421          20.857539   
min                   0.000000                0.000000           0.000000   
25%                   0.000000                0.000000           0.000000   
50%                   0.343427                2.153846           0.000000   
75%                   6.259055               14.894468           5.068981   
max                 269.496957              351.501987         534.600000   

       pluto_commercial_share  
count             1136.000000  
mean                 2.002153  
std                  4.464387  
min                  0.000000  
25%                  0.048615  
50%                  0.326285  
75%                  1.877523  
max                

In [ ]:
# Check Total Households before deciding to drop
print(df['Total Households'].describe())
print(f'Missing: {df["Total Households"].isnull().sum()}')

count    2224.000000
mean     1484.721223
std       894.734993
min         3.000000
25%       845.750000
50%      1308.000000
75%      1902.250000
max      8209.000000
Name: Total Households, dtype: float64
Missing: 100


In [ ]:
cols_to_drop = [

    # Duplicate income variable — keep median_income from ACS which is more standard
    'Household Income',

    # Individual race breakdowns from LEAD — keep pct_nonwhite from ACS instead
    'Black/ African American (% pop.)',
    'American Indian/ Native Alaskan (% pop.)',
    'Asian (% pop.)',
    'Native Hawaiian/ Other Pacific Islander (% pop.)',

    # Education columns from LEAD — not part of our feature set
    'Less Than High School (% pop.)',
    'High School (% pop.)',
    'Associates or Some College (% pop.)',
    'Bachelors or Higher (% pop.)',

    # Socioeconomic variables from canopy dataset — duplicated by ACS columns
    'canopy_pctpoc',
    'canopy_pctpov',
    'canopy_unemplrate',
    'pluto_pct_historic',
    'pluto_pct_landmark',
    'canopy_tes',
    'pluto_mean_effective_year',
    'pluto_median_yearbuilt',
    'canopy_health_nor',
    'canopy_tc_gap',
    'children_per_hh',


    # Raw children count — replaced by children_per_hh rate computed above
    'children_count',

    # Raw PLUTO totals — scale with tract size, replaced by normalized rates above
    'pluto_total_bldgarea',
    'pluto_total_resarea',
    'pluto_total_lotarea',
    'pluto_total_comarea',
    'pluto_total_unitsres',
    'pluto_total_unitstotal',
    'pluto_total_numbldgs',
    'pluto_parcel_count',

]

cols_to_drop = [c for c in cols_to_drop if c in df.columns]
df = df.drop(columns=cols_to_drop)
print(f'Shape after dropping: {df.shape}')
print()
print('Remaining columns:')
for col in df.columns:
    print(f'  {col}')

Shape after dropping: (2324, 29)

Remaining columns:
  GEOID
  Name
  Energy Burden (% income)
  Energy Burden (% income) (Electricity)
  Energy Burden (% income) (Gas)
  Energy Burden (% income) (Other)
  Avg. Annual Energy Cost ($)
  Avg. Annual Energy Cost ($) (Electricity)
  Avg. Annual Energy Cost ($) (Gas)
  Avg. Annual Energy Cost ($) (Other)
  Total Households
  Borough
  subway_05mi
  bus_025mi
  median_income
  pct_nonwhite
  pct_renter
  avg_hh_size
  pct_rent_burdened
  pluto_mean_builtfar
  pluto_mean_yearbuilt
  pluto_mean_numfloors
  pluto_pct_residential
  canopy_treecanopy
  canopy_temp_diff
  pluto_bldgarea_per_unit
  pluto_resarea_per_unit
  pluto_avg_lotarea
  pluto_commercial_share


In [ ]:
# Final check
print(f'Final shape: {df.shape[0]:,} rows x {df.shape[1]} columns')
print()
print('Missing values:')
missing = pd.DataFrame({
    'missing_count': df.isnull().sum(),
    'missing_pct': (df.isnull().sum() / len(df) * 100).round(1)
}).sort_values('missing_pct', ascending=False)
print(missing[missing['missing_count'] > 0].to_string())
print()
print('All columns:')
for col in df.columns:
    print(f'  {col}')

Final shape: 2,324 rows x 29 columns

Missing values:
                                           missing_count  missing_pct
pluto_commercial_share                              1188         51.1
pluto_bldgarea_per_unit                              119          5.1
pluto_resarea_per_unit                               119          5.1
Energy Burden (% income) (Gas)                       102          4.4
Energy Burden (% income)                             102          4.4
avg_hh_size                                          103          4.4
pct_renter                                           103          4.4
pct_nonwhite                                         103          4.4
pct_rent_burdened                                    103          4.4
median_income                                        103          4.4
Energy Burden (% income) (Electricity)               102          4.4
Energy Burden (% income) (Other)                     102          4.4
Avg. Annual Energy Cost ($)         

PREPARE FOR ML METHODS CHECKS

In [ ]:
# 1. Check your target variable
print(df['Energy Burden (% income)'].describe())
print(f'Missing: {df["Energy Burden (% income)"].isnull().sum()}')

# 2. Drop rows where target is missing - can't train without it
before = len(df)
df = df.dropna(subset=['Energy Burden (% income)'])
print(f'Dropped {before - len(df)} rows with missing target')
print(f'Rows remaining: {len(df):,}')

count    2222.000000
mean        2.787129
std         1.288664
min         0.000000
25%         2.000000
50%         3.000000
75%         4.000000
max        12.000000
Name: Energy Burden (% income), dtype: float64
Missing: 102
Dropped 102 rows with missing target
Rows remaining: 2,222


In [ ]:
feature_cols = [
    # Urban form
    'pluto_mean_yearbuilt',
    'pluto_mean_builtfar',
    'pluto_mean_numfloors',
    'pluto_bldgarea_per_unit',
    'pluto_resarea_per_unit',
    'pluto_avg_lotarea',
    'pluto_commercial_share',

    # Canopy / environment
    'canopy_treecanopy',
    'canopy_temp_diff',

    # Transit
    'subway_05mi',
    'bus_025mi',

    # Socioeconomic
    'median_income',
    'pct_nonwhite',
    'pct_renter',
    'avg_hh_size',
    'pct_rent_burdened',
    'Total Households',
]

# Check all features exist
missing_features = [c for c in feature_cols if c not in df.columns]
print(f'Missing feature columns: {missing_features}')

X = df[feature_cols]
y = df['Energy Burden (% income)']

print(f'\nX shape: {X.shape}')
print(f'y shape: {y.shape}')

Missing feature columns: []

X shape: (2222, 17)
y shape: (2222,)


In [ ]:
# 4. Check missing values in features
print('Missing values in feature set:')
missing = X.isnull().sum()
print(missing[missing > 0])

Missing values in feature set:
pluto_mean_yearbuilt         15
pluto_mean_builtfar          14
pluto_mean_numfloors         15
pluto_bldgarea_per_unit      34
pluto_resarea_per_unit       34
pluto_avg_lotarea            14
pluto_commercial_share     1091
canopy_treecanopy             4
canopy_temp_diff              4
median_income                 9
pct_nonwhite                  9
pct_renter                    9
avg_hh_size                   9
pct_rent_burdened             9
dtype: int64


In [ ]:
from sklearn.impute import SimpleImputer

# Fix the SettingWithCopyWarning
X = df[feature_cols].copy()

# Then fill commercial share
X['pluto_commercial_share'] = X['pluto_commercial_share'].fillna(0)

# Impute remaining missing values with column median
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy='median')
X_imputed = imputer.fit_transform(X)
X = pd.DataFrame(X_imputed, columns=X.columns, index=X.index)

print(f'Missing values after imputation: {X.isnull().sum().sum()}')
print(f'X shape: {X.shape}')

Missing values after imputation: 0
X shape: (2222, 17)


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import KFold, cross_val_score, cross_val_predict
from sklearn.impute import SimpleImputer
from google.colab import drive

drive.mount('/content/drive')

df = pd.read_csv('/content/drive/MyDrive/Spring 2026/Machine Learning For Cities/MLC Final Project/data/V2merged_data_clean.csv')
print(f'Loaded: {df.shape[0]:,} rows x {df.shape[1]} columns')

# Feature definitions
feature_cols = [
    'pluto_mean_yearbuilt',
    'pluto_mean_builtfar',
    'pluto_mean_numfloors',
    'pluto_bldgarea_per_unit',
    'pluto_resarea_per_unit',
    'pluto_avg_lotarea',
    'pluto_commercial_share',
    'pluto_pct_residential',
    'canopy_treecanopy',
    'canopy_temp_diff',
    'subway_05mi',
    'bus_025mi',
    'median_income',
    'pct_nonwhite',
    'pct_renter',
    'avg_hh_size',
    'pct_rent_burdened',
    'Total Households',
]

socio_cols = [
    'median_income',
    'pct_nonwhite',
    'pct_renter',
    'avg_hh_size',
    'pct_rent_burdened',
    'Total Households',
]

# Target variable
# Drop rows with missing or zero energy burden (zeros are non-residential tracts)
df = df[df['Energy Burden (% income)'].notna()].copy()
df = df[df['Energy Burden (% income)'] > 0].copy()
df = df.reset_index(drop=True)
y = df['Energy Burden (% income)'].reset_index(drop=True)
print(f'Rows after removing missing/zero target: {len(y):,}')

# Clean sentinel values (-666666666 used to encode missing in source data)
# must replace with NaN or they corrupt the model
df_clean = df[feature_cols].copy()
sentinel_cols = [c for c in feature_cols if df_clean[c].min() < -1000]
for col in sentinel_cols:
    n_bad = (df_clean[col] < -1000).sum()
    df_clean[col] = df_clean[col].where(df_clean[col] > -1000, np.nan)
    print(f'  Replaced {n_bad} sentinel values in {col}')

# feature matrix X
X = df_clean.copy()
# commercial_share NaN means no commercial floor area so fill with 0 insteadof removing
X['pluto_commercial_share'] = X['pluto_commercial_share'].fillna(0)
imputer = SimpleImputer(strategy='median')
X = pd.DataFrame(imputer.fit_transform(X), columns=feature_cols)

# socioeconomic-only feature matrix X_socio
X_socio = df_clean[socio_cols].copy()
imputer_socio = SimpleImputer(strategy='median')
X_socio = pd.DataFrame(imputer_socio.fit_transform(X_socio), columns=socio_cols)

# checks
print(f'\nX shape:       {X.shape}')
print(f'X_socio shape: {X_socio.shape}')
print(f'y shape:       {y.shape}')
print(f'Missing in X:       {X.isnull().sum().sum()}')
print(f'Missing in X_socio: {X_socio.isnull().sum().sum()}')
print(f'Missing in y:       {y.isnull().sum()}')
print(f'\ny range: {y.min()} to {y.max()}')
print('\nready for modeling')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loaded: 2,212 rows x 37 columns
Rows after removing missing/zero target: 2,212
  Replaced 23 sentinel values in median_income
  Replaced 1 sentinel values in avg_hh_size

X shape:       (2212, 18)
X_socio shape: (2212, 6)
y shape:       (2212,)
Missing in X:       0
Missing in X_socio: 0
Missing in y:       0

y range: 1.0 to 12.0

ready for modeling


In [ ]:
# Save updated file back to Drive
output_path = '/content/drive/MyDrive/Spring 2026/Machine Learning For Cities/MLC Final Project/data/V2merged_data_clean.csv'
df.to_csv(output_path, index=False)
print(f'Saved: {output_path}')
print(f'Final shape: {df.shape[0]:,} rows x {df.shape[1]} columns')

Saved: /content/drive/MyDrive/Spring 2026/Machine Learning For Cities/MLC Final Project/data/V2merged_data_clean.csv
Final shape: 2,212 rows x 37 columns
